Car vs truc CNN model 90%

In [ ]:
import tensorflow as tf
import pandas as pd

from keras import Sequential, layers, callbacks, optimizers, applications
from keras.preprocessing import image_dataset_from_directory

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    # Currently, memory growth needs to be the same across GPUs
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Memory growth must be set before GPUs have been initialized
    print(e)

Preprocess data and create the training and validation split 80% / 20%.

In [ ]:

size = 299

batch_size = 5

seed = 123

#buffer = 1000
data_dir = "../../data/tests/car_vs_truck"

input_shape = [size,size,3]

ds_train = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    seed=seed,
    subset="training",
    image_size=(size,size),
    batch_size=batch_size,
    shuffle=True
)

ds_valid= image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    seed=seed,
    subset="validation",
    image_size=(size,size),
    batch_size=batch_size,
    shuffle=True
)

#class_names = ds_train.class_names # type: ignore
#num_classes = len(ds_train)
#print(class_names)

In [ ]:
%%capture
for image_batch, labels_batch in ds_train:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

In [ ]:
#memory errors
AUTOTUNE = tf.data.AUTOTUNE

ds_train = ds_train.cache().prefetch(buffer_size=AUTOTUNE)
ds_valid = ds_valid.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
padding = "same"

body = applications.Xception(weights="imagenet")
body.trainable = False
model_layers = [
    
    #Preprocessing
    layers.Rescaling(1./255),
    
    #layers.RandomFlip(mode="vertical"),
  
    # Body
    body,
    
    # Classifier head   
    layers.Flatten(),
    layers.Dense(1024,activation="relu"),
    layers.Dense(1, activation="sigmoid")
]


In [ ]:

model = Sequential(layers=model_layers)

early_stopping = callbacks.EarlyStopping(patience=5,
                                      min_delta=1e-3,restore_best_weights=True)

optimizer = optimizers.Adam(learning_rate=2e-3)
model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["binary_accuracy"]
)
model.name = "car_vs_truck"

Train the model

In [ ]:
history = model.fit(
    ds_train,
    validation_data=ds_valid,
    callbacks=[early_stopping],
    epochs=50,
)

In [ ]:

history_df = pd.DataFrame(history.history)
history_df.loc[:, ['loss', 'val_loss']].plot()

In [ ]:
model.summary()